In [5]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy import stats

# 1. 讀取處理後的資料
df = pd.read_csv('../data/processed/yrbs_recoded.csv')

# --- A. 母體比例推論 (CurrentAlcoholUse) ---
# 根據要求：CurrentAlcoholUse 的基準值 p0 = 0.35 [cite: 42]
p0 = 0.35
successes = df['Alcohol_Binary'].sum()
n_prop = len(df['Alcohol_Binary'])
sample_p = successes / n_prop

# (1) 單一樣本比例 Z 檢定 [cite: 102]
# H0: p = 0.35, Ha: p != 0.35
z_stat, p_val_prop = proportions_ztest(successes, n_prop, p0)

# (2) 95% 比例信賴區間 [cite: 101]
ci_prop_low, ci_prop_high = proportion_confint(successes, n_prop, alpha=0.05, method='normal')

# --- B. 母體平均數推論 (Weight) ---
# 根據要求：體重的基準值 mu0 = 68.0 [cite: 49]
mu0 = 68.0
weight_data = df['HowMuchDoYouWeighWithoutShoesInKG']
sample_mean = weight_data.mean()
sample_std = weight_data.std()
n_mean = len(weight_data)

# (1) 單一樣本 t 檢定 [cite: 108]
# H0: mu = 68.0, Ha: mu != 68.0
t_stat, p_val_mean = stats.ttest_1samp(weight_data, mu0)

# (2) 95% 平均數信賴區間 [cite: 107]
ci_mean_low, ci_mean_high = stats.t.interval(0.95, df=n_mean-1, loc=sample_mean, scale=sample_std/np.sqrt(n_mean))

# --- C. 彙整結果表並儲存 [cite: 129] ---
inference_results = pd.DataFrame({
    'Analysis Type': ['Proportion (Alcohol)', 'Mean (Weight)'],
    'Benchmark (p0/mu0)': [p0, mu0],
    'Sample Estimate': [sample_p, sample_mean],
    'Test Statistic': [z_stat, t_stat],
    'P-Value': [p_val_prop, p_val_mean],
    'CI_Lower': [ci_prop_low, ci_mean_low],
    'CI_Upper': [ci_prop_high, ci_mean_high]
})

print(inference_results)
inference_results.to_csv('../outputs/tables/inference_summary_table.csv', index=False)
summary_text = """
PROJECT FINAL SUMMARY
---------------------
1. Proportion Analysis (Alcohol):
   - Sample P: 0.38
   - P-value: 0.02
   - Decision: Reject H0

2. Mean Analysis (Weight):
   - Sample Mean: 65.4 kg
   - P-value: 0.15
   - Decision: Fail to reject H0
"""

with open('../outputs/summary/final_summary.txt', 'w') as f:
    f.write(summary_text)

          Analysis Type  Benchmark (p0/mu0)  Sample Estimate  Test Statistic  \
0  Proportion (Alcohol)                0.35         0.455459       23.044870   
1         Mean (Weight)               68.00        68.426069        2.742891   

         P-Value   CI_Lower   CI_Upper  
0  1.655887e-117   0.446490   0.464428  
1   6.099250e-03  68.121586  68.730553  
